# 08. Llama 3.2 predictions for dataset C

Adds the `tool_predicted` column: the tool chosen by **Llama 3.2 3B**, run locally with Ollama
(`llama3.2:3b`), temperature 0, seed 42.

**Difference from datasets A and B.** There, the model was shown only the tools of the query's
role (attacker, defender or shared). Dataset C has no roles, so the model is shown **all 11
tools**, each with a one-line description. That is the same choice the classifier makes, and
tool descriptions are what an agent framework normally gives the model.

The model is also run on the 405 Claude-written test prompts.

**Replication:** predictions are saved to `llama_predictions_C.csv`. If that file exists the
notebook uses it (no Ollama needed). Set `RERUN = True` to call the model again.

In [1]:
import os, socket
import pandas as pd

C = pd.read_csv("cyber_dataset_C.csv")
T = pd.read_csv("cyber_testset_claude405.csv")
CACHE = "llama_predictions_C.csv"
MODEL = "llama3.2:3b"
RERUN = False      # True: call Llama again even if the saved predictions exist

TOOL_DESCRIPTIONS = {
    "SSHConnect":         "open an interactive SSH session on a remote host",
    "NmapScan":           "identify the services and software versions a remote host exposes",
    "PortScan":           "check which ports on a remote host are open or reachable",
    "CheckVulnerability": "look up known vulnerabilities, CVEs or advisories for a service or version",
    "ReadAuthLog":        "read the authentication log (login and session records) on this machine",
    "CheckFailedLogins":  "find and summarize failed or rejected login attempts",
    "ListeningPorts":     "list the ports and sockets this machine is listening on",
    "ListProcesses":      "list running processes and their CPU and memory use",
    "BlockIP":            "block or deny an IP address at the host firewall",
    "GetSystemInfo":      "show the hostname, OS distribution, kernel version and uptime",
    "ReadSyslog":         "read the general system log",
}
TOOLS = list(TOOL_DESCRIPTIONS)
SYSTEM_PROMPT = ("You are a tool selector for a cybersecurity agent. Given a task, respond with "
                 "exactly one word: the name of the single best tool. The tools are:\n" +
                 "\n".join(f"- {t}: {d}" for t, d in TOOL_DESCRIPTIONS.items()))

def ollama_up():
    s = socket.socket(); s.settimeout(2)
    ok = s.connect_ex(("127.0.0.1", 11434)) == 0
    s.close()
    return ok

def pick_tool(prompt):
    import ollama
    r = ollama.chat(model=MODEL,
                    messages=[{"role": "system", "content": SYSTEM_PROMPT},
                              {"role": "user", "content": prompt}],
                    options={"temperature": 0, "seed": 42})
    raw = r["message"]["content"]
    by_lower = {t.lower(): t for t in TOOLS}
    for tok in raw.replace(".", " ").replace(":", " ").replace(",", " ").replace("`", " ").replace("*", " ").split():
        if tok in TOOLS:
            return tok
        if tok.lower() in by_lower:
            return by_lower[tok.lower()]
    return "Unknown"

print(SYSTEM_PROMPT)
print()
print("Ollama running:", ollama_up())

You are a tool selector for a cybersecurity agent. Given a task, respond with exactly one word: the name of the single best tool. The tools are:
- SSHConnect: open an interactive SSH session on a remote host
- NmapScan: identify the services and software versions a remote host exposes
- PortScan: check which ports on a remote host are open or reachable
- CheckVulnerability: look up known vulnerabilities, CVEs or advisories for a service or version
- ReadAuthLog: read the authentication log (login and session records) on this machine
- CheckFailedLogins: find and summarize failed or rejected login attempts
- ListeningPorts: list the ports and sockets this machine is listening on
- ListProcesses: list running processes and their CPU and memory use
- BlockIP: block or deny an IP address at the host firewall
- GetSystemInfo: show the hostname, OS distribution, kernel version and uptime
- ReadSyslog: read the general system log

Ollama running: True


## 1. Smoke test on 5 rows

In [2]:
if ollama_up():
    for p, t in C.sample(5, random_state=0)[["prompt", "tool_ground_truth"]].values:   # live call
        print(f"{pick_tool(p):18} (annotated {t:18}) {p[:80]}")
else:
    print("Ollama not running: skipping the smoke test")

BlockIP            (annotated BlockIP           ) We've validated the alert and approved the emergency response. Drop requests fro


ReadSyslog         (annotated ReadSyslog        ) A storage warning appeared after yesterday's reboot. Show the general system mes


PortScan           (annotated PortScan          ) I suspect an accidental exposure on the development Redis node. Verify if 6379 i
GetSystemInfo      (annotated ListeningPorts    ) Is the management API listening only on localhost?


ListProcesses      (annotated ListProcesses     ) Are any defunct processes piling up?


## 2. Predict every prompt (dataset C and the Claude test set)

In [3]:
ALL = pd.concat([C[["row_id", "prompt"]], T[["row_id", "prompt"]]], ignore_index=True)
if os.path.exists(CACHE) and not RERUN:
    cache = pd.read_csv(CACHE)
    print("using saved predictions from", CACHE)
elif ollama_up():
    preds = []
    for i, p in enumerate(ALL["prompt"], 1):
        preds.append(pick_tool(p))
        if i % 200 == 0:
            print(f"  {i}/{len(ALL)}")
    cache = pd.DataFrame({"row_id": ALL["row_id"], "tool_predicted": preds})
    cache.to_csv(CACHE, index=False)
    print("saved", CACHE, "with", len(cache), "predictions")
else:
    raise SystemExit("Ollama is not running and there is no cached predictions file.")

lookup = dict(zip(cache["row_id"], cache["tool_predicted"]))
C["tool_predicted"] = C["row_id"].map(lookup)
T["tool_predicted"] = T["row_id"].map(lookup)
assert C["tool_predicted"].notna().all() and T["tool_predicted"].notna().all()
print("all", len(C), "+", len(T), "rows have a tool_predicted")

using saved predictions from llama_predictions_C.csv
all 1020 + 405 rows have a tool_predicted


## 3. How well does Llama do?

In [4]:
from sklearn.metrics import f1_score
for name, D in [("dataset C (1020)", C), ("Claude test set (405)", T)]:
    ok = D.tool_predicted == D.tool_ground_truth
    print(f"{name:22} accuracy {ok.mean():.1%} | macro F1 {f1_score(D.tool_ground_truth, D.tool_predicted, average='macro', labels=TOOLS):.3f}"
          f" | Unknown answers {int((D.tool_predicted == 'Unknown').sum())}")
print()
print("dataset C by source:")
print(C.assign(ok=C.tool_predicted == C.tool_ground_truth).groupby("source").ok.agg(n="size", acc="mean")
      .assign(acc=lambda d: (d.acc * 100).round(1)).to_string())
print()
print("dataset C by annotated tool:")
print(C.assign(ok=C.tool_predicted == C.tool_ground_truth).groupby("tool_ground_truth").ok.agg(n="size", acc="mean")
      .assign(acc=lambda d: (d.acc * 100).round(1)).to_string())

dataset C (1020)       accuracy 91.2% | macro F1 0.910 | Unknown answers 6
Claude test set (405)  accuracy 92.8% | macro F1 0.932 | Unknown answers 2

dataset C by source:
            n   acc
source             
batch440  440  91.4
batch550  550  90.9
pilot      30  93.3

dataset C by annotated tool:
                     n    acc
tool_ground_truth            
BlockIP             93  100.0
CheckFailedLogins   93   97.8
CheckVulnerability  93   96.8
GetSystemInfo       92  100.0
ListProcesses       93   82.8
ListeningPorts      93   46.2
NmapScan            92   94.6
PortScan            93   89.2
ReadAuthLog         93   97.8
ReadSyslog          93   98.9
SSHConnect          92   98.9


## 4. Save

In [5]:
C.to_csv("cyber_dataset_C.csv", index=False)
T.to_csv("cyber_testset_claude405.csv", index=False)
print("saved both files with tool_predicted")

saved both files with tool_predicted
